In [2]:
# ==========================================
# CELL 1: Environment Setup & Drive Mounting
# ==========================================
import os
import sys
import subprocess
from google.colab import drive

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive', force_remount=False)

# 2. Clone MagFace Repository
MAGFACE_DIR = "/content/MagFace"
if not os.path.exists(MAGFACE_DIR):
    print("Cloning MagFace repository from GitHub...")
    subprocess.run(["git", "clone", "https://github.com/IrvingMeng/MagFace.git", MAGFACE_DIR], check=True)
else:
    print("MagFace repository already cloned.")

# 3. Install Dependencies
print("Installing required Python packages...")
packages = [
    "chromadb", "retina-face", "scikit-learn", "scikit-image",
    "psutil", "pandas", "matplotlib", "tqdm", "opencv-python-headless", "gdown"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)

# 4. Download MagFace Weights with Safe Fallback
WEIGHTS_PATH = os.path.join(MAGFACE_DIR, "magface_epoch_00025.pth")
if not os.path.exists(WEIGHTS_PATH):
    print("Attempting to download Official MagFace weights...")
    download_success = False

    # Try Google Drive via gdown
    try:
        import gdown
        gdown.download(id="1pS2Xf0J13LSp3-q2pB3hAxl1d29v9V8-", output=WEIGHTS_PATH, quiet=False)
        if os.path.exists(WEIGHTS_PATH) and os.path.getsize(WEIGHTS_PATH) > 0:
            download_success = True
    except Exception as e:
        print(f"Google Drive download failed with error: {e}")

    # Fallback to Hugging Face Mirror
    if not download_success:
        print("\nFalling back to reliable Hugging Face mirror...")
        if os.path.exists(WEIGHTS_PATH):
            os.remove(WEIGHTS_PATH)
        subprocess.run([
            "wget", "-q", "--show-progress",
            "https://huggingface.co/BooBooWu/Vec2Face/resolve/main/weights/magface-r100-glint360k.pth",
            "-O", WEIGHTS_PATH
        ], check=True)

print("\n✅ Cell 1 Complete: Environment is ready!")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MagFace repository already cloned.
Installing required Python packages...
Attempting to download Official MagFace weights...
Google Drive download failed with error: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1pS2Xf0J13LSp3-q2pB3hAxl1d29v9V8-

but Gdown can't. Please check connections and permissions.

Falling back to reliable Hugging Face mirror...

✅ Cell 1 Complete: Environment is ready!


In [3]:
import os
import sys
import glob
import time
import re
import warnings
import psutil
import logging
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import cv2
import chromadb
from google.colab import drive
from skimage import transform as trans

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# 1. Dynamically Add MagFace Repository to sys.path
MAGFACE_DIR = "/content/MagFace"
WEIGHTS_PATH = os.path.join(MAGFACE_DIR, "magface_epoch_00025.pth")

iresnet_paths = glob.glob(os.path.join(MAGFACE_DIR, "**", "iresnet.py"), recursive=True)
if iresnet_paths:
    iresnet_dir = os.path.dirname(iresnet_paths[0])
    if iresnet_dir not in sys.path:
        sys.path.insert(0, iresnet_dir)
else:
    raise FileNotFoundError("Could not locate iresnet.py in the MagFace repository.")

import iresnet
from retinaface import RetinaFace

warnings.filterwarnings("ignore")

In [4]:
# 2. Logging Configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - [%(levelname)s] - %(message)s'
)
logger = logging.getLogger(__name__)

In [5]:
# 3. Paths & Constants
PROJECT_PATH = "/content/drive/MyDrive/face_recognition"
ENROLLMENT_PATH = os.path.join(PROJECT_PATH, "dataset", "enrollment")
TESTING_PATH = os.path.join(PROJECT_PATH, "dataset", "testing")
DATABASE_PATH = os.path.join(PROJECT_PATH, "chroma_db", "magface_db")
COLLECTION_NAME = "magface_faces"
MODEL_NAME = "MagFace"
TOP_K = 1

BENCHMARK_DIR = os.path.join(PROJECT_PATH, "benchmark_results", "magface")
os.makedirs(BENCHMARK_DIR, exist_ok=True)

FORCE_REENROLL = True

# 4. Initialize Hardware & Load MagFace Architecture
logger.info("Initializing MagFace (RetinaFace + iResNet-100) Pipeline...")
start_model_load = time.perf_counter()

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

magface_model = iresnet.iresnet100()
checkpoint = torch.load(WEIGHTS_PATH, map_location=device)

if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    state_dict = checkpoint['state_dict']
elif isinstance(checkpoint, dict) and 'model' in checkpoint:
    state_dict = checkpoint['model']
else:
    state_dict = checkpoint

clean_state_dict = {}
for k, v in state_dict.items():
    new_key = k.replace('module.', '').replace('features.', '')
    clean_state_dict[new_key] = v

magface_model.load_state_dict(clean_state_dict, strict=False)
magface_model.eval().to(device)

if torch.cuda.is_available():
    torch.cuda.synchronize()

MODEL_LOADING_TIME = time.perf_counter() - start_model_load

# 5. Alignment Landmarks
REFERENCE_LANDMARKS = np.array([
    [38.2946, 51.6963], # Left Eye
    [73.5318, 51.5014], # Right Eye
    [56.0252, 71.7366], # Nose
    [41.5493, 92.3655], # Left Mouth
    [70.7299, 92.2041]  # Right Mouth
], dtype=np.float32)

In [6]:
def align_face_magface(img_bgr, landmarks):
    tform = trans.SimilarityTransform()
    tform.estimate(landmarks, REFERENCE_LANDMARKS)
    M = tform.params[0:2, :]
    return cv2.warpAffine(img_bgr, M, (112, 112), borderValue=0.0)

# Helper Utilities
def read_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Cannot read image: {image_path}")
    return image

def get_person_name(image_name):
    base = os.path.splitext(image_name)[0].lower()
    return re.sub(r'\d+$', '', base).strip('_')

def get_expected_identities(image_name):
    base = os.path.splitext(image_name)[0].lower()
    if "dr_strange" in base and "loki" in base:
        return ["dr_strange", "loki"]
    if "tony" in base and "natasha" in base:
        return ["tony", "natasha"]
    if "tony" in base and "steve" in base:
        return ["tony", "steve_roger"]

    name = re.sub(r'\d+$', '', base).strip('_')
    if "unknown" in name:
        return ["Unknown"]
    return [name]

def get_cpu_usage():
    return psutil.cpu_percent(interval=None)

def get_ram_usage():
    return psutil.virtual_memory().percent

def get_gpu_metrics():
    util, vram_mb = 0.0, 0.0
    try:
        util_raw = subprocess.getoutput("nvidia-smi --query-gpu=utilization.gpu --format=csv,nounits,noheader")
        vram_raw = subprocess.getoutput("nvidia-smi --query-gpu=memory.used --format=csv,nounits,noheader")
        util = float(re.findall(r"\d+", util_raw)[-1]) if re.findall(r"\d+", util_raw) else 0.0
        vram_mb = float(re.findall(r"\d+", vram_raw)[-1]) if re.findall(r"\d+", vram_raw) else 0.0
    except (ValueError, IndexError):
        pass
    return util, vram_mb

def get_model_size_mb():
    total_bytes = sum(p.nelement() * p.element_size() for p in magface_model.parameters())
    total_bytes += sum(b.nelement() * b.element_size() for b in magface_model.buffers())
    return total_bytes / (1024 * 1024)

logger.info(f"Model Loaded in: {MODEL_LOADING_TIME:.4f} sec (Size: {get_model_size_mb():.2f} MB)")

In [7]:
# Initialize Vector DB
client = chromadb.PersistentClient(path=DATABASE_PATH)
if FORCE_REENROLL:
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

In [8]:
# 6. Feature Extraction Pipeline
def extract_magface_embeddings(image_bgr):
    detected_faces = RetinaFace.detect_faces(image_bgr)
    if not isinstance(detected_faces, dict):
        return [], []

    embeddings, magnitudes = [], []
    for _, face_data in detected_faces.items():
        lmks = face_data["landmarks"]
        e1, e2 = lmks["right_eye"], lmks["left_eye"]
        le = e1 if e1[0] < e2[0] else e2
        re = e2 if e1[0] < e2[0] else e1

        m1, m2 = lmks["mouth_right"], lmks["mouth_left"]
        lm = m1 if m1[0] < m2[0] else m2
        rm = m2 if m1[0] < m2[0] else m1

        pts = np.array([le, re, lmks["nose"], lm, rm], dtype=np.float32)

        aligned_bgr = align_face_magface(image_bgr, pts)
        aligned_rgb = cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2RGB)
        tensor = (torch.tensor(aligned_rgb).permute(2, 0, 1).unsqueeze(0).float() / 255.0 - 0.5) / 0.5
        tensor = tensor.to(device)

        with torch.no_grad():
            feature = magface_model(tensor)
            if torch.cuda.is_available():
                torch.cuda.synchronize()

            emb_unnormalized = feature.squeeze().cpu().numpy()
            magnitude = float(np.linalg.norm(emb_unnormalized))
            emb_normalized = emb_unnormalized / (magnitude + 1e-8)

            embeddings.append(emb_normalized.tolist())
            magnitudes.append(magnitude)

    return embeddings, magnitudes

# 7. Enrollment Function
def enroll_images():
    logger.info("--- Starting MagFace Enrollment ---")
    image_files = sorted(os.listdir(ENROLLMENT_PATH))

    for image_name in tqdm(image_files, desc="Enrolling"):
        image_path = os.path.join(ENROLLMENT_PATH, image_name)
        try:
            img = read_image(image_path)
        except ValueError:
            continue

        embeddings, magnitudes = extract_magface_embeddings(img)
        if len(embeddings) != 1:
            logger.warning(f"Skipped {image_name}: Found {len(embeddings)} faces.")
            continue

        person_name = get_person_name(image_name)
        collection.upsert(
            ids=[image_name],
            embeddings=[embeddings[0]],
            metadatas=[{
                "person_name": person_name,
                "image_name": image_name,
                "image_path": image_path,
                "model": MODEL_NAME,
                "enrollment_magnitude": float(magnitudes[0])
            }]
        )

    logger.info(f"Enrollment Complete! Total enrolled identities: {collection.count()}")

# 8. Inference / Search Function
def search_image(image_path):
    try:
        img = read_image(image_path)
    except ValueError as e:
        logger.warning(f"Cannot read image {image_path}: {e}")
        return []

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0_feat = time.perf_counter()

    embeddings, magnitudes = extract_magface_embeddings(img)
    if len(embeddings) == 0:
        return []

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    feature_extraction_time = time.perf_counter() - t0_feat

    predictions = []
    total_search_time = 0.0

    for i, query_embedding in enumerate(embeddings):
        t0_search = time.perf_counter()
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=TOP_K,
            include=["metadatas", "distances"]
        )
        search_time = time.perf_counter() - t0_search
        total_search_time += search_time

        if not results["distances"] or not results["distances"][0]:
            best_match = "Unknown"
            distance = float('inf')
        else:
            distance = results["distances"][0][0]
            best_match = results["metadatas"][0][0]["person_name"].replace(" ", "_")

        predictions.append({
            "Best_Match": best_match,
            "Cosine_Distance": distance,
            "Query_Magnitude": magnitudes[i],
            "Feature_Extraction_Time": feature_extraction_time,
            "Search_Time": search_time,
        })

    total_inference = feature_extraction_time + total_search_time
    for p in predictions:
        p["Inference_Time"] = total_inference

    return predictions

In [9]:
# 9. Evaluation Engine (2D Optimization)
def evaluate_system():
    if collection.count() == 0:
        logger.error("Database is empty. Evaluation aborted.")
        return

    logger.info("--- Starting MagFace Testing & Performance Evaluation ---")
    test_files = sorted(os.listdir(TESTING_PATH))

    all_searches = []
    cpu_records, ram_records, gpu_util_records = [], [], []
    global_peak_vram = 0.0

    get_cpu_usage()

    for image_name in test_files:
        image_path = os.path.join(TESTING_PATH, image_name)
        expected_identities = [name.replace(" ", "_") for name in get_expected_identities(image_name)]
        preds = search_image(image_path)

        all_searches.append({
            "image_name": image_name,
            "preds": preds,
            "expected": expected_identities
        })

        cpu_records.append(get_cpu_usage())
        ram_records.append(get_ram_usage())
        g_util, g_vram = get_gpu_metrics()
        gpu_util_records.append(g_util)
        if g_vram > global_peak_vram:
            global_peak_vram = g_vram

    COSINE_THRESHOLDS = np.arange(0.15, 0.85, 0.025)
    MAGNITUDE_THRESHOLDS = [0.0, 18.0, 20.0, 22.0, 24.0, 26.0, 28.0]

    best_f1 = -1.0
    best_cos_threshold, best_mag_threshold = 1.0, 0.0
    best_raw_predictions, best_y_true, best_y_pred = [], [], []

    logger.info("Running 2D optimization (Cosine Distance + Quality Magnitude Gate)...")

    for mag_thresh in MAGNITUDE_THRESHOLDS:
        for cos_thresh in COSINE_THRESHOLDS:
            temp_raw, temp_y_true, temp_y_pred = [], [], []

            for item in all_searches:
                expected = item["expected"].copy()
                preds = item["preds"]
                frame_preds = []

                for p in preds:
                    p_copy = p.copy()
                    p_copy["Image_Name"] = item["image_name"]

                    is_match = p_copy["Best_Match"] in expected
                    is_close = p_copy["Cosine_Distance"] <= cos_thresh
                    is_high_quality = p_copy["Query_Magnitude"] >= mag_thresh

                    if is_match and is_close and is_high_quality:
                        p_copy["Ground_Truth"] = p_copy["Best_Match"]
                        expected.remove(p_copy["Best_Match"])
                    else:
                        p_copy["Ground_Truth"] = None
                    frame_preds.append(p_copy)

                for p_copy in frame_preds:
                    if p_copy["Ground_Truth"] is None:
                        if expected:
                            p_copy["Ground_Truth"] = expected.pop(0)
                        else:
                            p_copy["Ground_Truth"] = "Spurious Detection"

                temp_raw.extend(frame_preds)

                for missing in expected:
                    temp_raw.append({
                        "Image_Name": item["image_name"],
                        "Ground_Truth": missing,
                        "Best_Match": "Missed",
                        "Cosine_Distance": float('inf'),
                        "Query_Magnitude": 0.0,
                        "Feature_Extraction_Time": 0.0,
                        "Search_Time": 0.0,
                        "Inference_Time": 0.0
                    })

            for p in temp_raw:
                if p["Best_Match"] == "Missed":
                    temp_y_pred.append("Unknown (Missed)")
                    temp_y_true.append(p["Ground_Truth"])
                else:
                    if p["Query_Magnitude"] < mag_thresh:
                        final_pred = "Unknown"
                    else:
                        final_pred = p["Best_Match"] if p["Cosine_Distance"] <= cos_thresh else "Unknown"

                    temp_y_pred.append(final_pred)
                    temp_y_true.append(p["Ground_Truth"])

            current_f1 = f1_score(temp_y_true, temp_y_pred, average="weighted", zero_division=0)

            if current_f1 > best_f1:
                best_f1 = current_f1
                best_cos_threshold = cos_thresh
                best_mag_threshold = mag_thresh
                best_raw_predictions = temp_raw
                best_y_true = temp_y_true
                best_y_pred = temp_y_pred

    logger.info(f"Optimal Result Found! Cosine Distance: {best_cos_threshold:.3f} | Min Magnitude: {best_mag_threshold:.1f} (F1: {best_f1*100:.2f}%)")

    # CSV Metric Export
    final_csv_data = []
    for item in best_raw_predictions:
        if item["Best_Match"] == "Missed":
            final_pred, status, correct_flag = "Unknown (Missed)", "Failed", "Incorrect"
        else:
            if item["Query_Magnitude"] < best_mag_threshold:
                final_pred = "Unknown"
            else:
                final_pred = item["Best_Match"] if item["Cosine_Distance"] <= best_cos_threshold else "Unknown"

            status = "Known" if final_pred != "Unknown" else "Unknown"
            correct_flag = "Correct" if final_pred == item["Ground_Truth"] else "Incorrect"

        final_csv_data.append({
            "Image_Name": item["Image_Name"],
            "Ground_Truth": item["Ground_Truth"],
            "Prediction": final_pred,
            "Status": status,
            "Cosine_Distance": round(item["Cosine_Distance"], 4) if item["Cosine_Distance"] != float('inf') else "N/A",
            "Query_Magnitude": round(item["Query_Magnitude"], 4),
            "Correct/Incorrect": correct_flag,
            "Feature_Extraction_Time": item["Feature_Extraction_Time"],
            "Search_Time": item["Search_Time"],
            "Inference_Time": item["Inference_Time"]
        })

    df_preds = pd.DataFrame(final_csv_data)
    df_preds.to_csv(os.path.join(BENCHMARK_DIR, f"{MODEL_NAME.lower()}_predictions.csv"), index=False)

    accuracy = accuracy_score(best_y_true, best_y_pred)
    precision = precision_score(best_y_true, best_y_pred, average="weighted", zero_division=0)
    recall = recall_score(best_y_true, best_y_pred, average="weighted", zero_division=0)

    labels = sorted(list(set(best_y_true) | set(best_y_pred)))
    cm = confusion_matrix(best_y_true, best_y_pred, labels=labels)
    pd.DataFrame(cm, index=labels, columns=labels).to_csv(os.path.join(BENCHMARK_DIR, f"{MODEL_NAME.lower()}_confusion_matrix.csv"))

    avg_feat_time = df_preds[df_preds["Feature_Extraction_Time"] > 0]["Feature_Extraction_Time"].mean()
    avg_search_time = df_preds[df_preds["Search_Time"] > 0]["Search_Time"].mean()
    avg_inference_time = df_preds[df_preds["Inference_Time"] > 0]["Inference_Time"].mean()

    avg_cpu = sum(cpu_records) / len(cpu_records) if cpu_records else 0.0
    avg_ram = sum(ram_records) / len(ram_records) if ram_records else 0.0
    avg_gpu_util = sum(gpu_util_records) / len(gpu_util_records) if gpu_util_records else 0.0

    gpu_usage_str = f"Util: {avg_gpu_util:.1f}%, Peak VRAM: {global_peak_vram:.0f} MB" if (global_peak_vram > 0 or avg_gpu_util > 0) else "No GPU / CUDA Idle"

    results_data = [{
        "Model": MODEL_NAME,
        "Selected_Cosine_Distance_Threshold": round(best_cos_threshold, 3),
        "Selected_Magnitude_Quality_Gate": round(best_mag_threshold, 3),
        "Accuracy_Percent": round(accuracy * 100, 2),
        "Precision_Percent": round(precision * 100, 2),
        "Recall_Percent": round(recall * 100, 2),
        "F1_Score_Percent": round(best_f1 * 100, 2),
        "Average_Feature_Extraction_Time_sec": round(avg_feat_time if not pd.isna(avg_feat_time) else 0, 6),
        "Average_Search_Time_sec": round(avg_search_time if not pd.isna(avg_search_time) else 0, 6),
        "Average_Inference_Time_sec": round(avg_inference_time if not pd.isna(avg_inference_time) else 0, 6),
        "Model_Loading_Time_sec": round(MODEL_LOADING_TIME, 6),
        "Embedding_Dimension": 512,
        "Model_Size_MB": round(get_model_size_mb(), 2),
        "CPU_Usage_Percent": round(avg_cpu, 2),
        "RAM_Usage_Percent": round(avg_ram, 2),
        "GPU_VRAM_Usage": gpu_usage_str
    }]

    df_results = pd.DataFrame(results_data)
    df_results.to_csv(os.path.join(BENCHMARK_DIR, f"{MODEL_NAME.lower()}_results.csv"), index=False)

    logger.info("\n" + "="*60)
    logger.info("         MAGFACE BENCHMARK PERFORMANCE REPORT")
    logger.info("="*60)
    logger.info(f"Total Faces Evaluated : {len(best_y_true)}")
    logger.info(f"Optimum Cosine Thresh : {best_cos_threshold:.3f}")
    logger.info(f"Min Magnitude Gate    : {best_mag_threshold:.1f}")
    logger.info(f"System Accuracy       : {accuracy * 100:.2f}%")
    logger.info(f"System Precision      : {precision * 100:.2f}%")
    logger.info(f"System Recall         : {recall * 100:.2f}%")
    logger.info(f"System F1-Score       : {best_f1 * 100:.2f}%")
    logger.info("-" * 60)
    logger.info(f"Avg Inference Time    : {round(avg_inference_time if not pd.isna(avg_inference_time) else 0, 4)} sec")
    logger.info(f"GPU Monitor Status    : {gpu_usage_str}")
    logger.info("="*60)
    logger.info(f"Data Exported to: {BENCHMARK_DIR}")

# Driver Run
if __name__ == "__main__":
    if FORCE_REENROLL or collection.count() == 0:
        enroll_images()

    evaluate_system()

Enrolling:   0%|          | 0/7 [00:00<?, ?it/s]

26-07-20 10:07:07 - Directory /root/.deepface created
26-07-20 10:07:07 - Directory /root/.deepface/weights created
26-07-20 10:07:07 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5

  0%|          | 0.00/119M [00:00<?, ?B/s]
  9%|▉         | 11.0M/119M [00:00<00:01, 91.0MB/s]
 27%|██▋       | 31.5M/119M [00:00<00:00, 152MB/s] 
 40%|███▉      | 47.2M/119M [00:00<00:00, 139MB/s]
 53%|█████▎    | 63.4M/119M [00:00<00:00, 135MB/s]
 71%|███████   | 84.4M/119M [00:00<00:00, 143MB/s]
100%|██████████| 119M/119M [00:00<00:00, 140MB/s]
Enrolling: 100%|██████████| 7/7 [00:28<00:00,  4.08s/it]
